## Overlay networks & the routing mesh

Module 05 met overlay networks as "the multi-host bridge." In Swarm they're **how every service talks across nodes**:

```bash
docker network create --driver overlay --attachable app-net
```

`--driver overlay` is required for multi-host; `--attachable` lets standalone containers join too (handy for debugging). Under the hood it's **VXLAN** — each cross-node packet is wrapped in a UDP datagram over port 4789 — so to the containers it looks like one flat LAN. Attach services and they reach each other **by service name**, cluster-wide (the module-05 embedded DNS, now spanning hosts): `web` calls `http://api` and Swarm resolves it. Tasks of one service share a **virtual IP**, and the kernel's IPVS load-balances across all healthy tasks wherever they run.

**The routing mesh** is the killer feature, hidden behind an ordinary `-p`. When a service publishes a port, Swarm opens that port on **every node in the cluster** — even nodes running no replica:

```
  Internet ─► Node A:8080 ──mesh──► task on Node C
          ─► Node B:8080 ──mesh──► task on Node A
          ─► Node C:8080 ──local─► task on Node C
```

A request to *any* `node:8080` is caught by IPVS, routed over the encrypted `ingress` overlay to a healthy task's node, and delivered locally. So you can put **any node's IP** behind your DNS or load balancer and it just works — the cluster handles internal routing. That's the default for `-p` in Swarm; if you'd rather publish only on nodes actually running a replica (e.g. to front it with an external LB), use **host-mode publishing** (`--publish mode=host`) instead. The routing mesh is what makes a Swarm service feel like one endpoint no matter how many machines it spans.